In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import matplotlib.colors as mcolors

from python.benchmark_constants import *

DATA_DIR = Path("./datasets")

sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.autolayout': True, 'figure.dpi': 120})

In [ ]:
from python.benchmark_io import load_benchmark_data, load_speedup_summary

df_bench = load_benchmark_data(
    DATA_DIR,
    time_field="time_s",
    statistic="median",
)

df_speedup = load_speedup_summary(
    DATA_DIR,
    time_field="time_per_iteration_s",
    ratio_statistic="median_ratio",
)

print(f"Loaded {len(df_bench)} summarized timing rows.")
print(f"Loaded {len(df_speedup)} post-processed speedup rows.")

display(df_bench.head())
display(df_speedup.head())

In [ ]:
from python.benchmark_validation import validate_lloyd_parity_parallel

df_parity = validate_lloyd_parity_parallel(data_dir=DATA_DIR, tolerance_pct=1e-4)
display(df_parity[df_parity["Status"] == "❌ FAIL"])

In [ ]:
from python.benchmark_transforms import *

df_bench = add_time_per_iteration_columns(df_bench)

df_lloyd = filter_bench(
    df_bench,
    phase=PHASE_MAP["lloyd"],
)

df_speedup = add_ci_error_columns(df_speedup)

df_speedup_lloyd = filter_bench(
    df_speedup,
    phase=PHASE_MAP["lloyd"],
)

In [ ]:
from python.benchmark_plotting import *

max_samples = df_bench[COL_SAMPLES].max()

df_math_summary = filter_bench(
    df_bench,
    phase=PHASE_MAP["lloyd"],
    language=LANG_CPP,
    samples=max_samples,
)[
    [COL_CLUSTERS, COL_DIMENSIONS, COL_TIME_PER_ITER]
].copy()

df_fixed_avg = filter_bench(
    df_bench,
    phase=[PHASE_MAP["soa"], PHASE_MAP["pp"]],
    language=LANG_CPP,
    samples=max_samples,
)[
    [COL_CLUSTERS, COL_DIMENSIONS, COL_PHASE, COL_TIME_S]
].copy()

df_plot = df_fixed_avg.merge(
    df_math_summary,
    on=[COL_CLUSTERS, COL_DIMENSIONS],
    validate="many_to_one",
)

df_plot[COL_EQUIVALENT_ITERS] = df_plot[COL_TIME_S] / df_plot[COL_TIME_PER_ITER]
df_plot[COL_PHASE] = df_plot[COL_PHASE].cat.remove_unused_categories()

unique_clusters = sorted(df_plot[COL_CLUSTERS].unique())
fig, axes = create_subplot_grid(
    len(unique_clusters),
    cols=2,
    row_height=5,
    fig_width=12,
)

for i, cluster in enumerate(unique_clusters):
    ax = axes[i]
    data_cluster = filter_bench(df_plot, clusters=cluster)

    sns.barplot(
        data=data_cluster,
        x=COL_DIMENSIONS,
        y=COL_EQUIVALENT_ITERS,
        hue=COL_PHASE,
        ax=ax,
    )

    ax.set_title(f"{cluster} Clusters", bbox=small_multiple_title_style)

    if i % 2 == 0:
        ax.set_ylabel("Cost (Equivalent Lloyd Iterations)")
    else:
        ax.set_ylabel("")

    ax.set_xlabel("Dimensions")
    ax.grid(axis="y", linestyle="--", alpha=0.6)

    if i != 0 and ax.get_legend() is not None:
        ax.get_legend().remove()

    ax2 = ax.twinx()
    ax2.grid(False)

    data_line = data_cluster.drop_duplicates(subset=[COL_DIMENSIONS]).copy()
    data_line[COL_TIME_PER_ITER_MS] = data_line[COL_TIME_PER_ITER] * 1000.0

    sns.pointplot(
        data=data_line,
        x=COL_DIMENSIONS,
        y=COL_TIME_PER_ITER_MS,
        ax=ax2,
        color="black",
        markersize=6,
        errorbar=None,
    )

    if i % 2 == 1:
        ax2.set_ylabel("Median Lloyd Time per Iteration (ms)", color="black")
    else:
        ax2.set_ylabel("")

    ax2.tick_params(axis="y", labelcolor="black")
    ax2.spines["right"].set_color("black")
    ax2.spines["right"].set_linewidth(1.4)

fig.suptitle(
    "Architectural Overhead: Fixed Costs expressed in Lloyd Iterations\n"
    f"Setup/memory cost at {format_abbrev(max_samples)} samples",
    fontsize=16,
    y=1,
)

plt.tight_layout()
plt.show()

In [ ]:
df_plot = df_speedup_lloyd.copy()
df_plot[COL_NBR_CLUSTERS] = df_plot[COL_CLUSTERS]

cluster_order = sorted(df_plot[COL_NBR_CLUSTERS].dropna().unique())

cluster_palette = dict(
    zip(
        cluster_order,
        sns.color_palette("Set1", n_colors=len(cluster_order)),
    )
)

g = sns.relplot(
    data=df_plot,
    x=COL_SAMPLES,
    y=COL_SPEEDUP,
    hue=COL_NBR_CLUSTERS,
    hue_order=cluster_order,
    col=COL_DIMENSIONS,
    col_wrap=3,
    kind="line",
    marker="o",
    palette=cluster_palette,
    height=4,
    aspect=1.2,
    facet_kws={"sharey": False},
)

for dim, ax in g.axes_dict.items():
    subset_dim = df_plot[df_plot[COL_DIMENSIONS] == dim]

    for cluster, subset in subset_dim.groupby(COL_NBR_CLUSTERS, observed=True):
        subset = subset.sort_values(COL_SAMPLES)

        x = subset[COL_SAMPLES].to_numpy()
        y_low = subset[COL_SPEEDUP_CI_LOW].to_numpy()
        y_high = subset[COL_SPEEDUP_CI_HIGH].to_numpy()

        ax.fill_between(
            x,
            y_low,
            y_high,
            color=cluster_palette[cluster],
            alpha=0.18,
            linewidth=0,
        )

style_facet_grid(
    g,
    title="Median Speedup vs. Sample Size with Clustered-Bootstrap CI\nPhase: Lloyd Iterations",
    title_y=1.02,
    x_log=True,
    sample_x_axis=True,
    titles_inside=True,
    grid_axis="both",
)

for ax in g.axes.flat:
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(bottom=0, top=ymax)

g.set_axis_labels("Number of Samples", "Speedup Multiplier (x)")


move_facet_legend_to_row_ends(
    g,
    default_title=COL_NBR_CLUSTERS,
    anchor=(1, 0.5),
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.patches as patches
from numpy.typing import NDArray

vmax_val = df_speedup_lloyd[COL_SPEEDUP].max()

cluster_vals = sorted(df_speedup_lloyd[COL_CLUSTERS].unique())
fig, axes = create_subplot_grid(len(cluster_vals), cols=2, row_height=6, fig_width=14)

for i, cluster in enumerate(cluster_vals):
    ax = axes[i]
    subset = filter_bench(df_speedup_lloyd, clusters=cluster)

    heatmap_pivot = subset.pivot(
        index=COL_DIMENSIONS,
        columns=COL_SAMPLES,
        values=COL_SPEEDUP,
    )

    ci_width_pivot = subset.pivot(
        index=COL_DIMENSIONS,
        columns=COL_SAMPLES,
        values=COL_SPEEDUP_ERROR_WIDTH,
    )

    heatmap_pivot.columns = [format_abbrev(col) for col in heatmap_pivot.columns]
    ci_width_pivot.columns = heatmap_pivot.columns

    show_cbar = i % 2 != 0
    cbar_kwargs = {
        "label": "Median Speedup Multiplier (x)",
        "format": mtick.ScalarFormatter(),
        "ticks": [1, 2, 5, 10, 20, 50, 100],
    } if show_cbar else {}

    cbar_ax = ax.inset_axes([1.04, 0, 0.05, 1]) if show_cbar else None

    sns.heatmap(
        heatmap_pivot,
        annot=True,
        fmt=".1f",
        cmap="turbo",
        norm=mcolors.LogNorm(vmin=1, vmax=vmax_val),
        vmin=1,
        vmax=vmax_val,
        ax=ax,
        cbar=show_cbar,
        cbar_ax=cbar_ax,
        cbar_kws=cbar_kwargs,
        linewidths=0.5,
    )

    relative_ci_pivot = ci_width_pivot / heatmap_pivot.abs()
    hatch_fraction_df = relative_ci_pivot.clip(lower=0, upper=1)

    hatch_fraction: NDArray[np.float64] = hatch_fraction_df.to_numpy(
        dtype=np.float64,
        na_value=np.nan,
    )

    for row_idx in range(hatch_fraction.shape[0]):
        for col_idx in range(hatch_fraction.shape[1]):
            frac = hatch_fraction[row_idx, col_idx]

            if np.isfinite(frac) and frac > 0:
                ax.add_patch(
                    patches.Rectangle(
                        (col_idx, row_idx),
                        1,
                        float(frac),
                        facecolor=(1, 1, 1, 0.08),
                        edgecolor=(0, 0, 0, 0.55),
                        hatch="////",
                        linewidth=0,
                        zorder=2,
                    )
                )

    for text in ax.texts:
        text.set_zorder(3)

    ax.set_title(f"Clusters: {cluster}", bbox=small_multiple_title_style)
    ax.set_xlabel("Number of Samples")
    ax.set_ylabel("Dimensions" if i % 2 == 0 else "")
    ax.tick_params(axis="x", labelrotation=0)

legend_handles = [
    patches.Patch(
        facecolor=(1, 1, 1, 0.08),
        edgecolor=(0, 0, 0, 0.55),
        hatch="////",
        label="Hatched area = relative CI width",
    )
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    frameon=False,
)

plt.suptitle(
    f"Median Speedup Heatmap ({LANG_CPP} vs. {LANG_PY}) by Number of Clusters\n"
    f"Phase: {PHASE_MAP['lloyd']}",
    y=1.0,
    fontsize=16,
    fontweight="bold",
)

plt.tight_layout()
plt.show()

In [ ]:
df_norm = add_speedup_retention(df_speedup)

base_k = df_norm[COL_CLUSTERS].min()
df_norm[COL_PHASE] = df_norm[COL_PHASE].cat.remove_unused_categories()

for phase in df_norm[COL_PHASE].unique():
    phase_data = filter_bench(df_norm, phase=phase)

    g = sns.relplot(
        data=phase_data,
        x=COL_CLUSTERS,
        y=COL_RETENTION,
        hue=COL_SAMPLES,
        col=COL_DIMENSIONS,
        col_wrap=3,
        kind="line",
        marker="o",
        hue_norm=mcolors.LogNorm(),
        legend="full",
        palette="turbo",
        height=4,
        aspect=1.2,
        facet_kws={"sharey": True},
    )

    style_facet_grid(
        g,
        title=(
            f"PHASE: {phase.upper()}\n"
            f"Speedup Retention against {base_k} Clusters\n"
            "Does C++ maintain its speed advantage over Python as K increases?"
        ),
        title_y=1.2,
        grid_axis="y",
        integer_x_axis=True,
    )

    g.set_axis_labels(COL_NBR_CLUSTERS, "Retention (%)")

    move_facet_legend_to_row_ends(
        g,
        default_title=COL_SAMPLES,
        label_formatter=format_abbrev,
        anchor=(1.05, 0.5),
    )

    for ax in g.axes.flat:
        ax.axhline(100, color="red", linestyle="--", alpha=0.5)

    plt.show()

In [ ]:
df_bench = add_time_per_iteration_per_sample_columns(
    df_bench,
    statistic="median",
    spread="iqr",
)

median_cluster_count = min(
    df_bench[COL_CLUSTERS].unique(),
    key=lambda x: abs(x - df_bench[COL_CLUSTERS].median()),
)

plot_df = filter_bench(
    df_bench,
    clusters=median_cluster_count,
    language=LANG_CPP,
    phase=phase,
)

plot_df[COL_PHASE] = plot_df[COL_PHASE].cat.remove_unused_categories()
plot_df = plot_df.sort_values([COL_DIMENSIONS, COL_SAMPLES])

def draw_spread_band(data, **kwargs):
    data = data.sort_values(COL_SAMPLES)

    ax = plt.gca()
    color = kwargs.get("color", sns.color_palette()[0])

    x = data[COL_SAMPLES].to_numpy()
    y_low = data[COL_TIME_PER_ITER_PER_SAMPLE_LOW_MS].to_numpy()
    y_high = data[COL_TIME_PER_ITER_PER_SAMPLE_HIGH_MS].to_numpy()

    sns.lineplot(
        data=data,
        x=COL_SAMPLES,
        y=COL_TIME_PER_ITER_PER_SAMPLE_MS,
        marker="o",
        errorbar=None,
        ax=ax,
        color=color,
    )

    ax.fill_between(
        x,
        y_low,
        y_high,
        color=color,
        alpha=0.2,
        linewidth=0,
    )

g = sns.FacetGrid(
    data=plot_df,
    col=COL_DIMENSIONS,
    col_wrap=3,
    sharey=False,
    height=4,
    aspect=1.2,
)

g.map_dataframe(draw_spread_band)

spread_label = plot_df[COL_RUN_TO_RUN_SPREAD].iloc[0]

style_facet_grid(
    g,
    title=(
        f"{LANG_CPP} median time per Lloyd iteration per sample vs. sample size\n"
        f"Error bars show run-to-run spread ({spread_label}); "
        f"{COL_NBR_CLUSTERS} = {median_cluster_count}"
    ),
    title_y=1.15,
    x_log=True,
    sample_x_axis=True,
)

g.set_axis_labels(
    "Number of samples",
    "Time per Lloyd iteration per sample (ms)",
)

plt.show()

In [ ]:
from scipy.optimize import least_squares
from sklearn.metrics import r2_score

COL_WORK_SIZE = "Work Size"
COL_SAMPLES_K = "Samples (K)"
COL_MODEL_NAME = "Model"
COL_NUM_PARAMS = "Num Params"
COL_PRED_TIME_PER_ITER_MS = "Predicted Time Per Iteration (ms)"
COL_PRED_TOTAL_TIME_MS = "Predicted Total Lloyd Time (ms)"
COL_TOTAL_TIME_MS = "Total Lloyd Time (ms)"
COL_ABS_ERROR_PCT = "Abs Error (%)"
COL_LOG_R2 = "Log-space R²"
COL_MEDIAN_ABS_ERROR_PCT = "Median Abs Error (%)"
COL_P90_ABS_ERROR_PCT = "P90 Abs Error (%)"
COL_MODEL_INTERCEPT_MS = "Intercept (ms)"
COL_MODEL_SLOPE_NS_PER_WORK = "Slope (ns / work unit)"
COL_TERM = "Term"
COL_COMPONENT = "Component"
COL_COEFFICIENT = "Coefficient"

COL_DIMENSIONS_SQUARED = f"{COL_DIMENSIONS}²"
COL_SAMPLES_K_X_DIMENSIONS = f"{COL_SAMPLES_K} × {COL_DIMENSIONS}"
COL_SAMPLES_K_X_DIMENSIONS_SQUARED = f"{COL_SAMPLES_K} × {COL_DIMENSIONS_SQUARED}"
COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS = (
    f"{COL_SAMPLES_K} × {COL_CLUSTERS} × {COL_DIMENSIONS}"
)
COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS_SQUARED = (
    f"{COL_SAMPLES_K} × {COL_CLUSTERS} × {COL_DIMENSIONS_SQUARED}"
)

CPP_MODEL_NAME = (
    "smooth-dimension model: "
    "S×D + S×K×D + S×K×D²"
)
PY_MODEL_NAME = (
    "amortized-overhead model: "
    "setup(1 + S + S×D) + iterations × iter(1 + S + S×D)"
)

df_model = df_lloyd.copy()

df_model[COL_TIME_PER_ITER_MS] = (
    df_model[COL_TIME_S]
    / df_model[COL_ITERATIONS]
    * 1_000
)

df_model[COL_TOTAL_TIME_MS] = df_model[COL_TIME_S] * 1_000

df_model[COL_WORK_SIZE] = (
    df_model[COL_SAMPLES]
    * df_model[COL_DIMENSIONS]
    * df_model[COL_CLUSTERS]
)

df_model[COL_SAMPLES_K] = df_model[COL_SAMPLES] / 1_000

df_model = df_model[
    df_model[COL_TIME_PER_ITER_MS].gt(0)
    & df_model[COL_TOTAL_TIME_MS].gt(0)
    & df_model[COL_WORK_SIZE].gt(0)
    & df_model[COL_SAMPLES_K].gt(0)
    & df_model[COL_ITERATIONS].gt(0)
].copy()


def _relative_error_metrics(y_true, y_pred):
    abs_error_pct = np.abs(y_pred / y_true - 1) * 100

    return {
        COL_MEDIAN_ABS_ERROR_PCT: np.median(abs_error_pct),
        COL_P90_ABS_ERROR_PCT: np.percentile(abs_error_pct, 90),
    }


def _fit_log_error_positive_linear_model(x, y):
    """Fit y = X @ beta by minimizing log-space error.

    Coefficients are constrained positive so predictions stay positive and the
    timing interpretation remains physically meaningful.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    initial_coef = np.linalg.lstsq(x, y, rcond=None)[0]
    initial_coef = np.maximum(initial_coef, 1e-12)

    def residuals(coef):
        y_hat = x @ coef
        return np.log(y_hat) - np.log(y)

    result = least_squares(
        residuals,
        x0=initial_coef,
        bounds=(np.full(x.shape[1], 1e-12), np.full(x.shape[1], np.inf)),
    )

    coef = result.x
    y_hat = x @ coef

    return coef, y_hat


# --- C++ model: no-overhead smooth-dimension model --------------------------

df_cpp = filter_bench(df_model, language=LANG_CPP).copy()

cpp_samples_k = df_cpp[COL_SAMPLES_K].to_numpy(dtype=float)
cpp_dimensions = df_cpp[COL_DIMENSIONS].to_numpy(dtype=float)
cpp_clusters = df_cpp[COL_CLUSTERS].to_numpy(dtype=float)

x_cpp = np.column_stack(
    [
        cpp_samples_k * cpp_dimensions,
        cpp_samples_k * cpp_clusters * cpp_dimensions,
        cpp_samples_k * cpp_clusters * cpp_dimensions**2,
    ]
)

cpp_terms = [
    COL_SAMPLES_K_X_DIMENSIONS,
    COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS,
    COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS_SQUARED,
]

y_cpp = df_cpp[COL_TIME_PER_ITER_MS].to_numpy(dtype=float)

cpp_coef, cpp_y_hat = _fit_log_error_positive_linear_model(x_cpp, y_cpp)

df_cpp[COL_PRED_TIME_PER_ITER_MS] = cpp_y_hat
df_cpp[COL_MODEL_NAME] = CPP_MODEL_NAME


# --- Scikit model: amortized-overhead model ---------------------------------

df_py = filter_bench(df_model, language=LANG_PY).copy()

py_samples_k = df_py[COL_SAMPLES_K].to_numpy(dtype=float)
py_dimensions = df_py[COL_DIMENSIONS].to_numpy(dtype=float)
py_iterations = df_py[COL_ITERATIONS].to_numpy(dtype=float)

x_py_base = np.column_stack(
    [
        np.ones(len(df_py)),
        py_samples_k,
        py_samples_k * py_dimensions,
    ]
)

x_py = np.column_stack(
    [
        x_py_base,
        x_py_base * py_iterations[:, None],
    ]
)

py_terms = [
    "Intercept",
    COL_SAMPLES_K,
    f"{COL_SAMPLES_K} × {COL_DIMENSIONS}",
]

y_py = df_py[COL_TOTAL_TIME_MS].to_numpy(dtype=float)

py_coef, py_total_y_hat = _fit_log_error_positive_linear_model(x_py, y_py)

n_py_terms = len(py_terms)
py_setup_coef = py_coef[:n_py_terms]
py_iter_coef = py_coef[n_py_terms:]

py_setup_y_hat = x_py_base @ py_setup_coef
py_iter_y_hat = x_py_base @ py_iter_coef
py_total_y_hat = py_setup_y_hat + py_iterations * py_iter_y_hat
py_y_hat = py_total_y_hat / py_iterations

df_py[COL_PRED_TOTAL_TIME_MS] = py_total_y_hat
df_py[COL_PRED_TIME_PER_ITER_MS] = py_y_hat
df_py[COL_MODEL_NAME] = PY_MODEL_NAME


# --- Summaries --------------------------------------------------------------

df_model_predictions = pd.concat([df_cpp, df_py], ignore_index=True)

summary_records = []

for language, model_name, df_lang, coef in [
    (LANG_CPP, CPP_MODEL_NAME, df_cpp, cpp_coef),
    (LANG_PY, PY_MODEL_NAME, df_py, py_coef),
]:
    y_true = df_lang[COL_TIME_PER_ITER_MS].to_numpy(dtype=float)
    y_pred = df_lang[COL_PRED_TIME_PER_ITER_MS].to_numpy(dtype=float)

    summary_records.append(
        {
            COL_LANGUAGE: language,
            COL_MODEL_NAME: model_name,
            COL_NUM_PARAMS: len(coef),
            COL_LOG_R2: r2_score(np.log(y_true), np.log(y_pred)),
            **_relative_error_metrics(y_true, y_pred),
        }
    )

df_model_summary = pd.DataFrame(summary_records)

df_model_coefficients = pd.concat(
    [
        pd.DataFrame(
            [
                {
                    COL_LANGUAGE: LANG_CPP,
                    COL_MODEL_NAME: CPP_MODEL_NAME,
                    COL_COMPONENT: "Per-iteration model",
                    COL_TERM: term,
                    COL_COEFFICIENT: coef,
                }
                for term, coef in zip(cpp_terms, cpp_coef)
            ]
        ),
        pd.DataFrame(
            [
                {
                    COL_LANGUAGE: LANG_PY,
                    COL_MODEL_NAME: PY_MODEL_NAME,
                    COL_COMPONENT: "Setup / amortized component",
                    COL_TERM: term,
                    COL_COEFFICIENT: coef,
                }
                for term, coef in zip(py_terms, py_setup_coef)
            ]
        ),
        pd.DataFrame(
            [
                {
                    COL_LANGUAGE: LANG_PY,
                    COL_MODEL_NAME: PY_MODEL_NAME,
                    COL_COMPONENT: "True per-iteration component",
                    COL_TERM: term,
                    COL_COEFFICIENT: coef,
                }
                for term, coef in zip(py_terms, py_iter_coef)
            ]
        ),
    ],
    ignore_index=True,
)

display(df_model_summary)
display(df_model_coefficients)


# --- Visualization ----------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 7))

sns.scatterplot(
    data=df_model_predictions,
    x=COL_WORK_SIZE,
    y=COL_TIME_PER_ITER_MS,
    hue=COL_LANGUAGE,
    alpha=0.35,
    s=35,
    ax=ax,
)

sns.scatterplot(
    data=df_cpp,
    x=COL_WORK_SIZE,
    y=COL_PRED_TIME_PER_ITER_MS,
    marker="X",
    s=70,
    color="black",
    alpha=0.05,
    label=f"{LANG_CPP} fitted smooth-dimension model",
    ax=ax,
)

sns.scatterplot(
    data=df_py,
    x=COL_WORK_SIZE,
    y=COL_PRED_TIME_PER_ITER_MS,
    marker="P",
    s=70,
    color="black",
    alpha=0.05,
    label=f"{LANG_PY} fitted amortized-overhead model",
    ax=ax,
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.xaxis.set_major_formatter(
    mtick.FuncFormatter(lambda x, _: format_abbrev(x))
)

ax.set_xlabel("Estimated Lloyd work size = samples × dimensions × clusters")
ax.set_ylabel("Time per Lloyd iteration (ms)")
ax.set_title(
    "Lloyd iteration timing models\n"
    "C++: no-overhead smooth-dimension model | "
    "Scikit-Learn: amortized-overhead model"
)

ax.grid(axis="both", linestyle="--", alpha=0.7)
ax.legend()

plt.tight_layout()

plt.show()

In [ ]:
# --- Cache sanity check for C++ residuals -----------------------------------

CACHE_FLOATS = {
    "L1 Data": 8_192,
    "L2": 131_072,
    "L3 Shared": 4_194_304,
}

COL_DATA_FLOATS = "Data Matrix Size (float32)"
COL_DATA_KIB = "Data Matrix Size (KiB)"
COL_CACHE_LEVEL = "Cache Level"
COL_CACHE_FLOATS = "Cache Size (float32)"
COL_CACHE_RATIO = "Data / Cache Size"
COL_LOG_RESIDUAL = "Log Residual"
COL_SIGNED_ERROR_PCT = "Signed Error (%)"

required_cols = [
    COL_SAMPLES,
    COL_DIMENSIONS,
    COL_CLUSTERS,
    COL_TIME_PER_ITER_MS,
    COL_PRED_TIME_PER_ITER_MS,
]

missing_cols = [col for col in required_cols if col not in df_cpp.columns]
if missing_cols:
    raise KeyError(f"df_cpp is missing required columns: {missing_cols}")

df_cpp_cache = df_cpp.copy()

df_cpp_cache[COL_DATA_FLOATS] = (
    df_cpp_cache[COL_SAMPLES]
    * df_cpp_cache[COL_DIMENSIONS]
)

df_cpp_cache[COL_DATA_KIB] = (
    df_cpp_cache[COL_DATA_FLOATS]
    * 4
    / 1024
)

df_cpp_cache[COL_LOG_RESIDUAL] = (
    np.log(df_cpp_cache[COL_TIME_PER_ITER_MS])
    - np.log(df_cpp_cache[COL_PRED_TIME_PER_ITER_MS])
)

df_cpp_cache[COL_SIGNED_ERROR_PCT] = (
    df_cpp_cache[COL_TIME_PER_ITER_MS]
    / df_cpp_cache[COL_PRED_TIME_PER_ITER_MS]
    - 1
) * 100

# --- Inspect large C++ residuals by dimension / samples / cache regime -------

HIGH_ERROR_THRESHOLD_PCT = 50

COL_ABS_SIGNED_ERROR_PCT = "Abs Signed Error (%)"
COL_EXCEEDS_L3 = "Exceeds L3"
COL_L3_RATIO = "Data / L3 Size"
COL_ERROR_BUCKET = "Error Bucket"

L3_FLOATS = CACHE_FLOATS["L3 Shared"]

df_cpp_cache_detail = df_cpp_cache.copy()

df_cpp_cache_detail[COL_L3_RATIO] = (
    df_cpp_cache_detail[COL_DATA_FLOATS] / L3_FLOATS
)

df_cpp_cache_detail[COL_EXCEEDS_L3] = (
    df_cpp_cache_detail[COL_L3_RATIO].gt(1)
)

df_cpp_cache_detail[COL_ABS_SIGNED_ERROR_PCT] = (
    df_cpp_cache_detail[COL_SIGNED_ERROR_PCT].abs()
)

df_cpp_cache_detail[COL_ERROR_BUCKET] = np.where(
    df_cpp_cache_detail[COL_SIGNED_ERROR_PCT].ge(HIGH_ERROR_THRESHOLD_PCT),
    f"Actual > predicted by {HIGH_ERROR_THRESHOLD_PCT}%+",
    np.where(
        df_cpp_cache_detail[COL_SIGNED_ERROR_PCT].le(-HIGH_ERROR_THRESHOLD_PCT),
        f"Actual < predicted by {HIGH_ERROR_THRESHOLD_PCT}%+",
        "Within ±50%",
    ),
)

display(
    df_cpp_cache_detail[
        df_cpp_cache_detail[COL_ABS_SIGNED_ERROR_PCT].ge(HIGH_ERROR_THRESHOLD_PCT)
    ]
    .sort_values(COL_SIGNED_ERROR_PCT, ascending=False)
    [
        [
            COL_DIMENSIONS,
            COL_SAMPLES,
            COL_CLUSTERS,
            COL_DATA_FLOATS,
            COL_L3_RATIO,
            COL_TIME_PER_ITER_MS,
            COL_PRED_TIME_PER_ITER_MS,
            COL_SIGNED_ERROR_PCT,
        ]
    ]
)

display(
    df_cpp_cache_detail
    .groupby([COL_DIMENSIONS, COL_EXCEEDS_L3], observed=True)
    .agg(
        Rows=(COL_SIGNED_ERROR_PCT, "count"),
        Median_Signed_Error_Pct=(COL_SIGNED_ERROR_PCT, "median"),
        Median_Abs_Error_Pct=(COL_SIGNED_ERROR_PCT, lambda s: np.median(np.abs(s))),
        P90_Abs_Error_Pct=(COL_SIGNED_ERROR_PCT, lambda s: np.percentile(np.abs(s), 90)),
        Max_Signed_Error_Pct=(COL_SIGNED_ERROR_PCT, "max"),
    )
    .reset_index()
    .sort_values([COL_EXCEEDS_L3, COL_DIMENSIONS])
)

fig, ax = plt.subplots(figsize=(12, 7))

sns.scatterplot(
    data=df_cpp_cache_detail,
    x=COL_L3_RATIO,
    y=COL_SIGNED_ERROR_PCT,
    hue=df_cpp_cache_detail[COL_DIMENSIONS].astype("category"), 
    style=COL_ERROR_BUCKET,
    size=COL_CLUSTERS,
    alpha=0.75,
    ax=ax,
)

ax.axhline(0, linestyle="--", alpha=0.7)
ax.axhline(HIGH_ERROR_THRESHOLD_PCT, linestyle=":", alpha=0.7)
ax.axhline(-HIGH_ERROR_THRESHOLD_PCT, linestyle=":", alpha=0.7)
ax.axvline(1, linestyle="--", alpha=0.9)

ax.set_xscale("log")
ax.set_xlabel("Data matrix size / L3 size")
ax.set_ylabel("Signed model error (%)")
ax.set_title(
    "Where does the C++ model break?\n"
    "Positive error = actual runtime is slower than predicted"
)

ax.grid(axis="both", linestyle="--", alpha=0.6)


plt.tight_layout()
plt.show()

In [ ]:
# --- C++ model search: per-sample + cache-aware no-overhead models -----------

from scipy.optimize import least_squares
from sklearn.metrics import r2_score

CACHE_FLOATS = {
    "L1 Data": 8_192,
    "L2": 131_072,
    "L3 Shared": 4_194_304,
}

COL_MODEL_FAMILY = "Model Family"
COL_DATA_FLOATS = "Data Matrix Size (float32)"
COL_DATA_KFLOATS = "Data Matrix Size (K float32)"
COL_L2_RATIO = "Data / L2 Size"
COL_L3_RATIO = "Data / L3 Size"
COL_OVER_L2_KFLOATS = "max(0, Data - L2) (K float32)"
COL_OVER_L3_KFLOATS = "max(0, Data - L3) (K float32)"
COL_OVER_L3_SAMPLES_K = "max(0, Data - L3) / D (K samples)"
COL_SIGNED_ERROR_PCT = "Signed Error (%)"
COL_ABS_ERROR_PCT = "Abs Error (%)"
COL_MEDIAN_ABS_ERROR_PCT = "Median Abs Error (%)"
COL_P90_ABS_ERROR_PCT = "P90 Abs Error (%)"
COL_P95_ABS_ERROR_PCT = "P95 Abs Error (%)"
COL_MAX_ABS_ERROR_PCT = "Max Abs Error (%)"
COL_LOG_R2 = "Log-space R²"
COL_NUM_PARAMS = "Num Params"
COL_TERM = "Term"
COL_COEFFICIENT = "Coefficient"

COL_SAMPLES_K = "Samples (K)"
COL_PRED_TIME_PER_ITER_MS = "Predicted Time Per Iteration (ms)"

COL_SAMPLES_K_X_CLUSTERS = f"{COL_SAMPLES_K} × {COL_CLUSTERS}"
COL_SAMPLES_K_X_DIMENSIONS = f"{COL_SAMPLES_K} × {COL_DIMENSIONS}"
COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS = (
    f"{COL_SAMPLES_K} × {COL_CLUSTERS} × {COL_DIMENSIONS}"
)
COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS_SQUARED = (
    f"{COL_SAMPLES_K} × {COL_CLUSTERS} × {COL_DIMENSIONS}²"
)

# Start from Lloyd rows only. This assumes df_lloyd already exists.
df_cpp_model_search = filter_bench(df_lloyd, language=LANG_CPP).copy()

df_cpp_model_search[COL_TIME_PER_ITER_MS] = (
    df_cpp_model_search[COL_TIME_S]
    / df_cpp_model_search[COL_ITERATIONS]
    * 1_000
)

df_cpp_model_search[COL_SAMPLES_K] = (
    df_cpp_model_search[COL_SAMPLES] / 1_000
)

df_cpp_model_search[COL_DATA_FLOATS] = (
    df_cpp_model_search[COL_SAMPLES]
    * df_cpp_model_search[COL_DIMENSIONS]
)

df_cpp_model_search[COL_DATA_KFLOATS] = (
    df_cpp_model_search[COL_DATA_FLOATS] / 1_000
)

df_cpp_model_search[COL_L2_RATIO] = (
    df_cpp_model_search[COL_DATA_FLOATS] / CACHE_FLOATS["L2"]
)

df_cpp_model_search[COL_L3_RATIO] = (
    df_cpp_model_search[COL_DATA_FLOATS] / CACHE_FLOATS["L3 Shared"]
)

df_cpp_model_search[COL_OVER_L2_KFLOATS] = np.maximum(
    0,
    df_cpp_model_search[COL_DATA_KFLOATS]
    - CACHE_FLOATS["L2"] / 1_000,
)

df_cpp_model_search[COL_OVER_L3_KFLOATS] = np.maximum(
    0,
    df_cpp_model_search[COL_DATA_KFLOATS]
    - CACHE_FLOATS["L3 Shared"] / 1_000,
)

df_cpp_model_search[COL_OVER_L3_SAMPLES_K] = (
    df_cpp_model_search[COL_OVER_L3_KFLOATS]
    / df_cpp_model_search[COL_DIMENSIONS]
)

df_cpp_model_search = df_cpp_model_search[
    df_cpp_model_search[COL_TIME_PER_ITER_MS].gt(0)
    & df_cpp_model_search[COL_SAMPLES_K].gt(0)
    & df_cpp_model_search[COL_DIMENSIONS].gt(0)
    & df_cpp_model_search[COL_CLUSTERS].gt(0)
].copy()


def _fit_log_error_positive_linear_model(x, y):
    """Fit y = X @ beta by minimizing log-space error.

    Coefficients are constrained positive so the timing model remains
    physically meaningful and predictions stay positive.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    initial_coef = np.linalg.lstsq(x, y, rcond=None)[0]
    initial_coef = np.maximum(initial_coef, 1e-12)

    def residuals(coef):
        y_hat = x @ coef
        return np.log(y_hat) - np.log(y)

    result = least_squares(
        residuals,
        x0=initial_coef,
        bounds=(
            np.full(x.shape[1], 1e-12),
            np.full(x.shape[1], np.inf),
        ),
    )

    coef = result.x
    y_hat = x @ coef

    return coef, y_hat


def _relative_error_summary(y_true, y_pred):
    signed_error_pct = (y_true / y_pred - 1) * 100
    abs_error_pct = np.abs(signed_error_pct)

    return {
        COL_LOG_R2: r2_score(np.log(y_true), np.log(y_pred)),
        COL_MEDIAN_ABS_ERROR_PCT: np.median(abs_error_pct),
        COL_P90_ABS_ERROR_PCT: np.percentile(abs_error_pct, 90),
        COL_P95_ABS_ERROR_PCT: np.percentile(abs_error_pct, 95),
        COL_MAX_ABS_ERROR_PCT: np.max(abs_error_pct),
    }


def _make_cpp_features(df, terms):
    samples_k = df[COL_SAMPLES_K].to_numpy(dtype=float)
    dimensions = df[COL_DIMENSIONS].to_numpy(dtype=float)
    clusters = df[COL_CLUSTERS].to_numpy(dtype=float)

    feature_map = {
        COL_SAMPLES_K: samples_k,
        COL_SAMPLES_K_X_DIMENSIONS: samples_k * dimensions,
        COL_SAMPLES_K_X_CLUSTERS: samples_k * clusters,
        COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS: samples_k * clusters * dimensions,
        COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS_SQUARED: (
            samples_k * clusters * dimensions**2
        ),
        COL_OVER_L2_KFLOATS: df[COL_OVER_L2_KFLOATS].to_numpy(dtype=float),
        COL_OVER_L3_KFLOATS: df[COL_OVER_L3_KFLOATS].to_numpy(dtype=float),
        COL_OVER_L3_SAMPLES_K: df[COL_OVER_L3_SAMPLES_K].to_numpy(dtype=float),
    }

    return np.column_stack([feature_map[term] for term in terms])


cpp_candidate_models = {
    "Baseline smooth-dimension": [
        COL_SAMPLES_K_X_DIMENSIONS,
        COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS,
        COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS_SQUARED,
    ],
    "Add per-sample": [
        COL_SAMPLES_K,
        COL_SAMPLES_K_X_DIMENSIONS,
        COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS,
        COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS_SQUARED,
    ],
    "Add per-sample and S×K": [
        COL_SAMPLES_K,
        COL_SAMPLES_K_X_DIMENSIONS,
        COL_SAMPLES_K_X_CLUSTERS,
        COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS,
        COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS_SQUARED,
    ],
    "Add L3 overflow terms": [
        COL_SAMPLES_K,
        COL_SAMPLES_K_X_DIMENSIONS,
        COL_SAMPLES_K_X_CLUSTERS,
        COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS,
        COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS_SQUARED,
        COL_OVER_L3_KFLOATS,
        COL_OVER_L3_SAMPLES_K,
    ],
    "Add L2 and L3 overflow terms": [
        COL_SAMPLES_K,
        COL_SAMPLES_K_X_DIMENSIONS,
        COL_SAMPLES_K_X_CLUSTERS,
        COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS,
        COL_SAMPLES_K_X_CLUSTERS_X_DIMENSIONS_SQUARED,
        COL_OVER_L2_KFLOATS,
        COL_OVER_L3_KFLOATS,
        COL_OVER_L3_SAMPLES_K,
    ],
}

y_cpp = df_cpp_model_search[COL_TIME_PER_ITER_MS].to_numpy(dtype=float)

summary_records = []
coefficient_records = []
prediction_frames = []

for model_name, terms in cpp_candidate_models.items():
    x_cpp = _make_cpp_features(df_cpp_model_search, terms)

    coef, y_hat = _fit_log_error_positive_linear_model(x_cpp, y_cpp)

    df_pred = df_cpp_model_search.copy()
    df_pred[COL_MODEL_FAMILY] = model_name
    df_pred[COL_PRED_TIME_PER_ITER_MS] = y_hat
    df_pred[COL_SIGNED_ERROR_PCT] = (
        df_pred[COL_TIME_PER_ITER_MS]
        / df_pred[COL_PRED_TIME_PER_ITER_MS]
        - 1
    ) * 100
    df_pred[COL_ABS_ERROR_PCT] = df_pred[COL_SIGNED_ERROR_PCT].abs()

    prediction_frames.append(df_pred)

    summary_records.append(
        {
            COL_MODEL_FAMILY: model_name,
            COL_NUM_PARAMS: len(terms),
            **_relative_error_summary(y_cpp, y_hat),
        }
    )

    coefficient_records.extend(
        {
            COL_MODEL_FAMILY: model_name,
            COL_TERM: term,
            COL_COEFFICIENT: term_coef,
        }
        for term, term_coef in zip(terms, coef)
    )

df_cpp_model_search_predictions = pd.concat(
    prediction_frames,
    ignore_index=True,
)

df_cpp_model_search_summary = (
    pd.DataFrame(summary_records)
    .sort_values([COL_P90_ABS_ERROR_PCT, COL_MEDIAN_ABS_ERROR_PCT])
    .reset_index(drop=True)
)

df_cpp_model_search_coefficients = pd.DataFrame(coefficient_records)

display(df_cpp_model_search_summary)
display(df_cpp_model_search_coefficients)

best_cpp_model_name = df_cpp_model_search_summary.loc[0, COL_MODEL_FAMILY]

df_cpp_best_model = df_cpp_model_search_predictions[
    df_cpp_model_search_predictions[COL_MODEL_FAMILY].eq(best_cpp_model_name)
].copy()

print(f"Best C++ model by P90 absolute error: {best_cpp_model_name}")

display(
    df_cpp_best_model
    .sort_values(COL_ABS_ERROR_PCT, ascending=False)
    [
        [
            COL_DIMENSIONS,
            COL_SAMPLES,
            COL_CLUSTERS,
            COL_L3_RATIO,
            COL_TIME_PER_ITER_MS,
            COL_PRED_TIME_PER_ITER_MS,
            COL_SIGNED_ERROR_PCT,
            COL_ABS_ERROR_PCT,
        ]
    ]
    .head(20)
)

fig, ax = plt.subplots(figsize=(12, 7))

sns.scatterplot(
    data=df_cpp_best_model,
    x=COL_PRED_TIME_PER_ITER_MS,
    y=COL_TIME_PER_ITER_MS,
    hue=df_cpp_best_model[COL_DIMENSIONS].astype("category"),
    size=COL_CLUSTERS,
    alpha=0.7,
    ax=ax,
)

min_time = min(
    df_cpp_best_model[COL_PRED_TIME_PER_ITER_MS].min(),
    df_cpp_best_model[COL_TIME_PER_ITER_MS].min(),
)
max_time = max(
    df_cpp_best_model[COL_PRED_TIME_PER_ITER_MS].max(),
    df_cpp_best_model[COL_TIME_PER_ITER_MS].max(),
)

ax.plot([min_time, max_time], [min_time, max_time], linestyle="--", alpha=0.8)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel("Predicted C++ time per Lloyd iteration (ms)")
ax.set_ylabel("Measured C++ time per Lloyd iteration (ms)")
ax.set_title(
    f"C++ no-overhead model fit\n"
    f"Best model: {best_cpp_model_name}"
)

ax.grid(axis="both", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()


fig, ax = plt.subplots(figsize=(12, 7))

sns.scatterplot(
    data=df_cpp_best_model,
    x=COL_L3_RATIO,
    y=COL_SIGNED_ERROR_PCT,
    hue=df_cpp_best_model[COL_DIMENSIONS].astype("category"),
    size=COL_CLUSTERS,
    alpha=0.75,
    ax=ax,
)

ax.axhline(0, linestyle="--", alpha=0.7)
ax.axvline(1, linestyle="--", alpha=0.9)

ax.set_xscale("log")

ax.set_xlabel("Data matrix size / L3 size")
ax.set_ylabel("Signed model error (%)")
ax.set_title(
    f"C++ residuals vs. L3 pressure\n"
    f"Best model: {best_cpp_model_name}"
)

ax.grid(axis="both", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()